In [1]:
import os,sys
print(os.getcwd())
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print("Project root directory:", project_root)
sys.path.append(project_root)

/home/zhaolei/meal-rec-ai/preprocess
Project root directory: /home/zhaolei/meal-rec-ai


In [2]:
from utils.data import concat_data_across_years,root_path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Dietary Supplement Data

In [3]:
def Supplmement():
    years = ['0708', '0910', '1112', '1314', '1516', '1718', '1720']
    year_char = 'E'
    type_dietary = 'dietary'
    df_suppl1 = concat_data_across_years(type_dietary, 'DS1TOT', years, year_char)
    df_suppl2 = concat_data_across_years(type_dietary, 'DS2TOT', years, year_char)
    df_suppl1 = df_suppl1.rename(columns={'DR1DAY': 'day'})
    df_suppl2 = df_suppl2.rename(columns={'DR2DAY': 'day'})
    
     #columns
    suppl_columns_1 = ['SEQN', 'day','DS1DS','years',
                      'DS1TKCAL', 'DS1TPROT', 'DS1TCARB', 'DS1TSUGR', 'DS1TFIBE',
                      'DS1TSFAT', 'DS1TCHOL', 'DS1TFA', 'DS1TVB12',
                      'DS1TVC', 'DS1TVD', 'DS1TCALC', 'DS1TPHOS', 'DS1TIRON', 'DS1TSODI',
                      ]
    suppl_columns_2 = ['SEQN', 'day','DS2DS','years',
                      'DS2TKCAL', 'DS2TPROT', 'DS2TCARB', 'DS2TSUGR', 'DS2TFIBE',
                      'DS2TSFAT', 'DS2TCHOL', 'DS2TFA', 'DS2TVB12',
                      'DS2TVC', 'DS2TVD', 'DS2TCALC', 'DS2TPHOS', 'DS2TIRON', 'DS2TSODI',
                      ]
    df_suppl1 = df_suppl1[suppl_columns_1].astype(float)
    df_suppl2 = df_suppl2[suppl_columns_2].astype(float)
    df_suppl = pd.DataFrame(np.vstack((df_suppl1.to_numpy(), df_suppl2.to_numpy())), columns=df_suppl1.columns)
    nutrition_mapping = {'DS1TKCAL': 'calorie', 'DS1TPROT': 'protein', 'DS1TCARB': 'carb', 'DS1TSUGR': 'sugar',
                         'DS1TFIBE': 'fiber',
                         'DS1TSFAT': 'saturated_fat', 'DS1TCHOL': 'cholesterol', 'DS1TSODI': 'sodium',
                         'DS1TCALC': 'calcium', 'DS1TPHOS': 'phosphorus',
                         'DS1TPOTA': 'potassium', 'DS1TIRON': 'iron', 'DS1TFA': 'folic_acid', 'DS1TVC': 'vitamin_c',
                         'DS1TVD': 'vitamin_d', 'DS1TVB12': 'vitamin_b12'
                         }
    df_suppl = df_suppl.rename(columns=nutrition_mapping)
    df_suppl[['SEQN','years']] = df_suppl[['SEQN','years']].astype(int).astype(str)
    return df_suppl

In [4]:
df_suppl=Supplmement()

In [5]:
print(len(df_suppl))
df_suppl = df_suppl[df_suppl['DS1DS']==1]
df_suppl=df_suppl.drop(columns=['DS1DS'])
print(len(df_suppl))

143428
39556


In [6]:
df_suppl = df_suppl.fillna(0)
df_suppl['years'] = df_suppl['years'].apply(lambda x: x.zfill(4))
df_suppl = df_suppl.rename(columns={'SEQN': 'user_id'})

In [7]:
df_suppl.to_csv(f'{root_path}supplement.csv',index=False)

In [8]:
df_suppl

,user_id,day,years,calorie,protein,carb,sugar,fiber,saturated_fat,cholesterol,folic_acid,vitamin_b12,vitamin_c,vitamin_d,calcium,phosphorus,iron,sodium
0,41475,5.0,0708,10.0,0.0,0.0,0.0,0.0,0.0,0.0,200.0,25.0,150.0,5.0,700.0,0.0,0.0,0.0
1,41476,7.0,0708,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,160.0,0.0,0.0,0.0,0.0,0.0
4,41479,6.0,0708,0.0,0.0,0.0,0.0,0.0,0.0,0.0,400.0,18.0,90.0,10.0,210.0,0.0,0.0,0.0
7,41482,6.0,0708,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,600.0,0.0,0.0,0.0
8,41483,5.0,0708,0.0,0.0,0.0,0.0,0.0,0.0,0.0,500.0,6.0,90.0,10.0,200.0,109.0,18.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
143392,124785,2.0,1720,10.0,0.5,0.0,0.0,0.0,0.0,5.0,400.0,50.0,100.0,25.0,300.0,20.0,8.0,0.0
143412,124806,4.0,1720,30.0,0.0,4.0,4.0,0.0,0.0,0.0,400.0,6.0,50.0,50.0,0.0,0.0,0.0,20.0
143417,124812,4.0,1720,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,1200.0,0.0,0.0,0.0
143423,124818,3.0,1720,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1000.0,12.0,180.0,20.0,400.0,218.0,32.0,0.0


# Physical Activity

In [9]:
def pa():
    years = ['0304','0506','0708', '0910', '1112', '1314', '1516', '1718', '1720']
    year_char = 'C'
    type_dietary = 'questionnaire'
    df_pa = concat_data_across_years(type_dietary, 'PAQ', years, year_char)
    df_pa = df_pa.rename(columns={'PAQ180': 'level'})
    
    return df_pa

In [10]:
df_pa=pa()

In [11]:
df_pa=df_pa[df_pa['level']>=0]
df_pa = df_pa[['SEQN','level','years']]
df_pa[['SEQN','level','years']] = df_pa[['SEQN','level','years']].astype(int).astype(str)

In [12]:
df_pa['years'] = df_pa['years'].apply(lambda x: x.zfill(4))
df_pa = df_pa.rename(columns={'SEQN': 'user_id'})

In [13]:
df_pa.to_csv(f'{root_path}pa.csv',index=False)

In [14]:
df_pa

,user_id,level,years
0,21005,3,0304
1,21006,1,0304
3,21008,3,0304
4,21009,3,0304
5,21010,2,0304
...,...,...,...
9418,41468,2,0506
9419,41469,2,0506
9421,41472,2,0506
9422,41473,3,0506
